# Properties cold calling list with score

# Project Overview

## Objective

The goal of this project is to identify **Single Family Homes** in Anne Arundel County, Maryland, that have a higher likelihood of being sold.

## Data Processing

* Retrieved property records from the county's ArcGIS REST API.
* Filtered the dataset to include only **Single Family Homes** (SFD and SFA).
* Standardized structure type values to ensure consistency across the dataset.
* Renamed and formatted fields to improve readability and simplify analysis.

## Analysis

The analysis focuses on identifying properties that are more likely to be listed for sale based on historical ownership data.

One of the primary indicators is the **Last Transfer Date**. Properties that have not changed ownership for **3–4 years or more** are considered more likely to be sold compared to recently transferred properties.

Additional analysis includes:

* Identifying the cities with the highest concentration of eligible properties.
* Determining geographic areas where these properties are clustered.
* Summarizing the distribution of potential opportunities across the county.

## Expected Outcome

The final dataset provides a filtered list of Single Family Homes along with supporting information that can be used for lead generation, market analysis, or further predictive modeling.


## Project steps
1- Extracting the data from public county gis

2- Cleaning the data 

In [3]:
import requests
import pandas as pd
import math
import numpy as np
from bs4 import BeautifulSoup
import os

In [4]:
BASE_URL = "https://gis.aacounty.org/arcgis/rest/services/OpenData/Structure_OpenData/MapServer/0/query"

# Number of records retrieved in each  request
BATCH_SIZE = 1000

# Starting point for pagination
offset = 0

# Output CSV file
csv_file = "MD_Anne_Arnuld_GDSC.csv"

# Continue requesting data until returns no more records
while True:
    params = {
        "f": "json",
        "where": "1=1",
        "outFields": "*",
        "resultRecordCount": BATCH_SIZE,
        "resultOffset": offset,
        "orderByFields": "OBJECTID ASC"
    }
    
# Sending request 
    response = requests.get(BASE_URL, params=params)
    data = response.json()

# Extract the list of features from the response
    features = data.get("features", [])
    
# Stop the loop when no more records are available
    if not features:
        print("No more data to fetch.")
        break
        
# Convert feature attributes into a DataFrame       
    df = pd.json_normalize([f["attributes"] for f in features])
    
# Append the current batch to the CSV file
# Write the header only when the file is created for the first time
    df.to_csv(
        csv_file,
        mode="a",
        index=False,
        header=not os.path.exists(csv_file),
        encoding="utf-8-sig"
    )
    print(f"Added {len(df)} records.")
    
   # Move to the next batch
    offset += BATCH_SIZE

print(f"All data saved to {csv_file}")

Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.
Added 1000 records.


In [39]:
# saving DataFrame as csv with low_memory = Flase to handle the mixes datatypes columns
df = pd.read_csv("MD_Anne_Arnuld_GDSC.csv", low_memory = False)

In [40]:
# to display all the columns without limit
pd.set_option('display.max_columns', None)
df.head(3)

,OBJECTID,ADDRPT_ID,MASTER_ID,PT_TYPE,FULL_ADDRE,ST_NUMBER,ST_NUMSUFF,ST_PREFIXD,ST_NAME,ST_TYPE,ST_SUFFIXD,BLDG_NAME,SOURCE,CITY_NAME,UNIT_TYPE,UNITNUM,FLOOR,UNIT_ADDR,SUB_ADDR,BUS_NAME,AA_POI,ZIPCODE,AAID,AA_CP,BMC_ID,ACCESS_X,ACCESS_Y,BLDG_CODE,BLDG_ID,RESTRICTED,CAD_ST_TYP,CAD_ADDR,COMMENTS,POINT_X,POINT_Y,STRUC_TYPE,GlobalID,created_us,created_da,last_edite,last_edi_1
0,1,6001687,0,ADDRESS,"625 IRVIN AVE, 20751",625,,,IRVIN,AVE,,,PHYSICALLY VERIFIED IN FIELD,DEALE,,,0,,,,N,20751,6371,N,1,0.0,0.0,RESIDENTIAL,,0,AV,625 IRVIN AV,ADDED PT_TYPE,1439283.965,402495.2904,SFD,{2C227459-6D66-4968-9676-BB2BD456CF67},,1.220000e+12,GIS_AnneArundel,1.740000e+12
1,2,6001692,0,ADDRESS,"626 IRVIN AVE, 20751",626,,,IRVIN,AVE,,,PHYSICALLY VERIFIED IN FIELD,DEALE,,,0,,,,N,20751,6371,N,1,0.0,0.0,RESIDENTIAL,,0,AV,626 IRVIN AV,,1439211.749,402767.2886,SFD,{1FF38C66-6BD9-4750-ABF7-B80327084922},,1.220000e+12,pzmagl22,1.740000e+12
2,3,6001700,0,ADDRESS,"615 IRVIN AVE, 20751",615,,,IRVIN,AVE,,,PHYSICALLY VERIFIED IN FIELD,DEALE,,,0,,,,N,20751,6371,N,1,0.0,0.0,RESIDENTIAL,,0,AV,615 IRVIN AV,,1439111.054,402678.7467,SFD,{CA218AD6-6785-4FC1-9B24-CF3945C6DB13},,1.220000e+12,ITMAGL23,1.600000e+12


In [41]:
# Changing the data type for these columns from ms to date time
df["created_da"] = pd.to_datetime(df["created_da"], unit="ms", errors="coerce")
df["last_edi_1"] = pd.to_datetime(df["last_edi_1"], unit="ms", errors="coerce")

In [42]:
# renaming the column with more Descriptive Name
df.rename(columns={"last_edi_1": "Transfer_Date"}, inplace=True)

In [44]:
# to be sure the date time type changed and the column name
df.head()

,OBJECTID,ADDRPT_ID,MASTER_ID,PT_TYPE,FULL_ADDRE,ST_NUMBER,ST_NUMSUFF,ST_PREFIXD,ST_NAME,ST_TYPE,ST_SUFFIXD,BLDG_NAME,SOURCE,CITY_NAME,UNIT_TYPE,UNITNUM,FLOOR,UNIT_ADDR,SUB_ADDR,BUS_NAME,AA_POI,ZIPCODE,AAID,AA_CP,BMC_ID,ACCESS_X,ACCESS_Y,BLDG_CODE,BLDG_ID,RESTRICTED,CAD_ST_TYP,CAD_ADDR,COMMENTS,POINT_X,POINT_Y,STRUC_TYPE,GlobalID,created_us,created_da,last_edite,Transfer_Date
0,1,6001687,0,ADDRESS,"625 IRVIN AVE, 20751",625,,,IRVIN,AVE,,,PHYSICALLY VERIFIED IN FIELD,DEALE,,,0,,,,N,20751,6371,N,1,0.0,0.0,RESIDENTIAL,,0,AV,625 IRVIN AV,ADDED PT_TYPE,1439283.965,402495.2904,SFD,{2C227459-6D66-4968-9676-BB2BD456CF67},,2008-08-29 08:53:20,GIS_AnneArundel,2025-02-19 21:20:00
1,2,6001692,0,ADDRESS,"626 IRVIN AVE, 20751",626,,,IRVIN,AVE,,,PHYSICALLY VERIFIED IN FIELD,DEALE,,,0,,,,N,20751,6371,N,1,0.0,0.0,RESIDENTIAL,,0,AV,626 IRVIN AV,,1439211.749,402767.2886,SFD,{1FF38C66-6BD9-4750-ABF7-B80327084922},,2008-08-29 08:53:20,pzmagl22,2025-02-19 21:20:00
2,3,6001700,0,ADDRESS,"615 IRVIN AVE, 20751",615,,,IRVIN,AVE,,,PHYSICALLY VERIFIED IN FIELD,DEALE,,,0,,,,N,20751,6371,N,1,0.0,0.0,RESIDENTIAL,,0,AV,615 IRVIN AV,,1439111.054,402678.7467,SFD,{CA218AD6-6785-4FC1-9B24-CF3945C6DB13},,2008-08-29 08:53:20,ITMAGL23,2020-09-13 12:26:40
3,4,6001704,0,ADDRESS,"622 IRVIN AVE, 20751",622,,,IRVIN,AVE,,,PHYSICALLY VERIFIED IN FIELD,DEALE,,,0,,,,N,20751,6371,N,1,0.0,0.0,RESIDENTIAL,,0,AV,622 IRVIN AV,,1439271.443,402690.8779,SFD,{157B0DA6-9491-4072-A6DD-E6FBBCE7453D},,2008-08-29 08:53:20,ITMAGL23,2020-09-13 12:26:40
4,5,6001709,0,ADDRESS,"624 IRVIN AVE, 20751",624,,,IRVIN,AVE,,,PHYSICALLY VERIFIED IN FIELD,DEALE,,,0,,,,N,20751,6371,N,1,0.0,0.0,RESIDENTIAL,,0,AV,624 IRVIN AV,,1439239.348,402729.9582,SFD,{2A3346E6-FA35-4F68-BCFA-C33AC1CF097F},,2008-08-29 08:53:20,ITMAGL23,2020-09-13 12:26:40


In [45]:
# to make cleaning data easier and know what we need and what we do not
columns = ["BLDG_CODE","ST_SUFFIXD","PT_TYPE","UNIT_ADDR", "UNIT_TYPE", "UNITNUM","UNIT_ADDR","SUB_ADDR","BUS_NAME","ACCESS_X","STRUC_TYPE"]

for col in columns:
    print(f"\n=== {col} ===")
    print(df[col].value_counts())


=== BLDG_CODE ===
BLDG_CODE
RESIDENTIAL            390874
COMMERCIAL              14808
PLANNED                  9326
GOVERNMENT               5650
LAND                     4326
OPEN SPACE               2072
ASSET                    1257
PLACE OF WORSHIP          766
EDUCATIONAL               406
RECREATIONAL AREA         390
OTHER INSTITUTIONAL       372
MEDICAL                   190
PARKING                   128
MIXED USE                  98
FIRE STATION               80
POST OFFICE                60
LIBRARY                    28
STORAGE                    26
AIRPORT                    24
CORRECTIONAL               18
POLICE STATION              8
NO STRUCTURE                8
Name: count, dtype: int64

=== ST_SUFFIXD ===
ST_SUFFIXD
      419193
W       2930
E       2580
S       1830
N       1762
SW      1090
SE       536
NE       532
NW       462
Name: count, dtype: int64

=== PT_TYPE ===
PT_TYPE
ADDRESS    430849
COMPLEX        66
Name: count, dtype: int64

=== UNIT_ADDR ===
UNIT_

In [46]:
# we need only Adresses And RESIDENTIAL Code 
df = df[df["PT_TYPE"] == "ADDRESS"]
df = df[df["BLDG_CODE"] == "RESIDENTIAL"]

In [47]:
# saving STRUC_TYPE with only single family homes
df = df[
    df["STRUC_TYPE"].isin([
        "SFD",
        "SFA",
        "SINGLE FAMILY DWELLING",
        "SINGLE FAMILY ATTACHED"
    ])
]

In [48]:
# Replace verbose structure type names with standardized abbreviations
# to keep the dataset consistent for filtering and analysis.
df["STRUC_TYPE"] = df["STRUC_TYPE"].replace({
    "SINGLE FAMILY DWELLING": "SFD",
    "SINGLE FAMILY ATTACHED": "SFA"
})